# Notebook 02: RetailMart 360 — Data Cleaning & Quality Pipeline

**Project:** RetailMart 360 — Retail Sales & Customer Intelligence  
**Role:** Senior Data Analyst & Analytics Consultant  
**Environment:** Python 3.13 | Pandas | Data Engineering for Analytics  

---

## 1. Business Objective
This notebook demonstrates the end-to-end data transformation pipeline that converts raw CSV database dumps into validated, standardized analytical datasets. 

### Cleaning Directives:
1. Standardize text strings and strip whitespace.
2. Fix the `offices.territory` ingestion bug (map NaN to `'NA'`).
3. Parse and standardize all date columns to ISO format.
4. Derive order-line commercial metrics: `revenue`, `estimatedCost`, `estimatedGrossProfit`, `estimatedGrossMarginPct`.
5. Derive operational SLA metrics: `shippingDays`, `isLateShipment`, `is_shipped`.
6. Construct the denormalized `sales_analytical_mart` and validate before-and-after data quality.


In [1]:
import os
import pandas as pd
import numpy as np

# Load raw tables and cleaning module
from src.data_loader import load_all_raw_tables
from src.data_cleaning import run_pipeline

raw_tables = load_all_raw_tables()
print("Raw tables loaded successfully.")


Raw tables loaded successfully.


## 2. Resolving the Offices Territory Parsing Bug
In raw `offices.csv`, the territory code `'NA'` represents North America, but standard parsers interpret it as missing (`NaN`). We inspect and resolve this.


In [2]:
raw_offices = raw_tables["offices"]
print("Raw offices territory before fix:")
print(raw_offices[["officeCode", "city", "country", "territory"]])

# Fix territory
clean_offices_df = raw_offices.copy()
clean_offices_df["territory"] = clean_offices_df["territory"].fillna("NA").astype(str).str.strip()
print("\nCleaned offices territory:")
print(clean_offices_df[["officeCode", "city", "country", "territory"]])


Raw offices territory before fix:
   officeCode           city    country territory
0           1  San Francisco        USA       NaN
1           2         Boston        USA       NaN
2           3            NYC        USA       NaN
3           4          Paris     France      EMEA
4           5          Tokyo      Japan     Japan
5           6         Sydney  Australia      APAC
6           7         London         UK      EMEA

Cleaned offices territory:
   officeCode           city    country territory
0           1  San Francisco        USA        NA
1           2         Boston        USA        NA
2           3            NYC        USA        NA
3           4          Paris     France      EMEA
4           5          Tokyo      Japan     Japan
5           6         Sydney  Australia      APAC
6           7         London         UK      EMEA


## 3. Deriving Order Line Revenue, Cost, and Profit Margins
We compute realized line-item revenue as $\text{quantityOrdered} \times \text{priceEach}$, merge factory `buyPrice`, and compute gross contribution profit.


In [3]:
od = raw_tables["orderdetails"].copy()
prod = raw_tables["products"].copy()

od["revenue"] = (od["quantityOrdered"] * od["priceEach"]).round(2)
od = od.merge(prod[["productCode", "buyPrice", "MSRP", "productName", "productLine"]], on="productCode", how="left")
od["estimatedCost"] = (od["quantityOrdered"] * od["buyPrice"]).round(2)
od["estimatedGrossProfit"] = (od["revenue"] - od["estimatedCost"]).round(2)
od["estimatedGrossMarginPct"] = ((od["estimatedGrossProfit"] / od["revenue"]) * 100).round(2)

print("Sample Cleaned Order Details with Commercial Metrics:")
display(od[["orderNumber", "productCode", "quantityOrdered", "priceEach", "revenue", "estimatedGrossProfit", "estimatedGrossMarginPct"]].head())


orderNumber productCode  quantityOrdered  priceEach  revenue  estimatedGrossProfit  estimatedGrossMarginPct
0        10100    S18_1749               30     136.00  4080.00               1499.70                    36.76
1        10100    S18_2248               50      55.09  2754.50               1070.00                    38.85
2        10100    S18_4409               22      75.46  1660.12                679.58                    40.94
3        10100    S24_3969               49      35.29  1729.21                417.48                    24.14
4        10101    S18_2325               25     108.06  2701.50                778.00                    28.80


## 4. Executing the Production Cleaning Pipeline
We run the centralized pipeline to output all cleaned tables and analytical data marts.


In [4]:
clean_tables, sales_mart, customer_rfm = run_pipeline()
print(f"Pipeline executed successfully!")
print(f"Sales Mart Rows: {len(sales_mart)} | Columns: {len(sales_mart.columns)}")
print(f"Customer RFM Rows: {len(customer_rfm)} | Columns: {len(customer_rfm.columns)}")


Saved offices_clean: 7 rows -> C:\Users\pkatt\OneDrive\Desktop\Projects\RetailMart_Analytics\data\processed\offices_clean.csv
Saved employees_clean: 23 rows -> C:\Users\pkatt\OneDrive\Desktop\Projects\RetailMart_Analytics\data\processed\employees_clean.csv
Saved customers_clean: 122 rows -> C:\Users\pkatt\OneDrive\Desktop\Projects\RetailMart_Analytics\data\processed\customers_clean.csv
Saved products_clean: 110 rows -> C:\Users\pkatt\OneDrive\Desktop\Projects\RetailMart_Analytics\data\processed\products_clean.csv
Saved productlines_clean: 7 rows -> C:\Users\pkatt\OneDrive\Desktop\Projects\RetailMart_Analytics\data\processed\productlines_clean.csv
Saved orders_clean: 326 rows -> C:\Users\pkatt\OneDrive\Desktop\Projects\RetailMart_Analytics\data\processed\orders_clean.csv
Saved orderdetails_clean: 2996 rows -> C:\Users\pkatt\OneDrive\Desktop\Projects\RetailMart_Analytics\data\processed\orderdetails_clean.csv
Saved payments_clean: 273 rows -> C:\Users\pkatt\OneDrive\Desktop\Projects\Retai